In [27]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor as VIF   # 2-4에서 사용

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 170)
np.set_printoptions(suppress=True)

In [28]:
DATA, BOOK = 'knhanes_like_week4.csv', 'codebook_week4.csv'
df = pd.read_csv(DATA)
w  = df['wt'].to_numpy(float)
ST, PS = df['kstrata'].to_numpy(), df['psu'].to_numpy()

print(f'n = {len(df):,}  |  조사구 {df.psu.nunique()}개  |  층 {df.kstrata.nunique()}개')
print(f'가중치 합 = {w.sum():,.0f}')

n = 5,367  |  조사구 263개  |  층 18개
가중치 합 = 450,899


#노출·결과 확정 + DAG 작성
💬 노출 = 교육수준(edu) · 결과 = 고혈압(htn) · 추정 대상 (총효과 / 직접효과) = 총효과 (흡연이 체중·혈압 등 모든 경로를 통해 고혈압에 미치는 전체 영향을 알고 싶으므로 매개경로를 막지 않는다.)

💬 DAG — 손그림 사진을 넣거나 화살표 목록으로 적으십시오. 노출·결과를 포함해 최소 6개 변수, 순환 없음. 방향에 확신이 없는 화살표에는 * 를 붙이고 아래에 이유를 적으십시오.

```
age → edu              age → htn
male → edu *           male → htn
urban → edu *          urban → htn
edu → htn              (직접경로)
edu → income           income → htn
edu → smk_cur          smk_cur → htn
edu → drink            drink → htn
edu → exercise         exercise → htn
edu → bmi              bmi → htn
edu → sbp              sbp → htn          bmi → sbp
sbp → htn_med          htn_med → htn      (htn 정의에 약물복용 포함)
bmi → prog             htn → prog         (교육용 변수 prog)
height                 (어느 변수와도 인과 화살표 없음)
```
변수 15개(노출·결과 포함), 순환 없음.

**`*` 화살표의 이유**
- male -> edu * (성별 $\rightarrow$ 교육수준)이유: 생물학적 성별은 선천적으로 결정되어 교육보다 선행하지만, 세대별(코호트) 사회적 환경이나 문화적 배경에 따라 성별에 따른 교육 기회 격차 양상이 달라질 수 있으며, 단순한 단방향 인과로 단정하기에는 사회구조적 맥락의 불확실성이 존재함.
- urban -> edu * (거주지역 $\rightarrow$ 교육수준)이유: 도시 지역의 우수한 교육 인프라가 교육 수준에 영향을 미칠 수 있으나(정방향), 반대로 고등교육 이수나 학업을 위해 도시로 인구가 유입되는 역방향 인과관계(이주)의 가능성도 공존하므로 인과 방향에 양방향적 불확실성이 있음.

#후보변수 네 유형 분류
| 변수 | 유형 (교란/매개/충돌부/무관) | 근거 (한 줄) |
|---|---|---|
| `age` | 교란 | 출생 시기(세대)가 교육 기회를 정하고(고령일수록 학력이 낮음) 연령 자체가 혈압을 올린다 (edu ← age → htn). 자료에서도 연령–교육 상관 −0.52 |
| `male` | 교란 * | 고령 세대에서는 성별 교육 격차가 있었으므로 edu ← male → htn 가능. 단 이 자료에서 성별–교육 상관은 0.006으로 거의 없다 |
| `urban` | 교란 또는 매개 * | 성장 지역이 교육 기회를 정하면 교란, 교육 후 거주지를 옮겼다면 매개. |
| `income` | 매개 | 교육 → 소득 → 고혈압 (소득은 교육 이후에 결정되므로 교란일 수 없다) |
| `smk_cur, drink, exercise` | 매개 | 교육 → 건강행동 → 고혈압. 다만 이 자료에서는 교육과 거의 무관하다(흡연율 22.7–25.5%로 평평) |
| `bmi` | 매개 | 교육 → 체중 → 고혈압 |
| `sbp` | 매개 | 교육 → 혈압 → 고혈압. 게다가 고혈압 정의에 들어 있어 사실상 결과의 일부 |
| `htn_med` | 매개 | 고혈압 정의에 포함 → 모형에 넣으면 완전 분리(htn_med=1이면 모두 htn=1)가 생겨 보정 자체가 불가능 |
| `prog` | 충돌부 | 비만(bmi)도 고혈압도 참여를 높인다 (bmi → prog ← htn) |
| `height` | 무관 | 교육의 결과도 고혈압의 원인도 아니다 (교육과 상관 −0.005) |

#보정집합 결정과 이유
💬 보정집합 = { age, male }

💬 넣은 변수와 뺀 변수 각각의 이유 (한 문단 이상)

> **넣은 변수.** 연령은 노출과 결과의 가장 강한 공통 원인이다. 이 자료에서 보정 전후 초졸 이하 vs 대졸 이상 OR이 6.34에서 1.13으로 변한다. 성별은 세대별 교육 격차가 있을 수 있는 교란 후보이며, 해가 없으므로(매개도 충돌부도 아님) 포함한다. 연령을 직선·이차항으로 넣은 결과가 같아 연령 보정은 충분하다.

> **뺀 변수.** 소득·흡연·음주·운동·체중·혈압은 교육 이후의 변수(매개)라서 보정하면 총효과의 일부를 지운다. htn_med와 sbp는 결과의 정의에도 들어 있어 더욱 안 된다. prog는 충돌부(고혈압의 후손)라서 보정하면 막혀 있던 경로가 열린다. height는 무관해서 넣을 이유가 없다. 이 셋이 B-3(과보정 시연)의 재료다.

> **최소충분성.** 위 DAG에서 {age, male}이 충분하며 male → edu 화살표를 지우면 {age}만으로도 충분하다. 두 경우의 결과가 같음을 민감도 분석으로 확인했다

In [29]:
# 팀 과제 — 복합표본 회귀
#   노출 = 교육수준(edu) · 결과 = 고혈압(htn) · 총효과
#   보정집합 = {age, male}
import numpy as np
import pandas as pd
import statsmodels.api as sm

# 1. 데이터 복사 및 결측치 처리
dfp = df.copy()

if 'edu' in dfp.columns:
    dfp['edu'] = dfp['edu'].replace(9, np.nan)
if 'income' in dfp.columns:
    dfp['income'] = dfp['income'].replace(9, np.nan)

n0 = len(dfp)
check_cols = [c for c in ['edu', 'income'] if c in dfp.columns]
if check_cols:
    dfp = dfp.dropna(subset=check_cols)

print(f"→ 분석 대상 n = {len(dfp):,}")

wp = dfp['wt'].to_numpy(float) if 'wt' in dfp.columns else np.ones(len(dfp))
stp = dfp['kstrata'].to_numpy() if 'kstrata' in dfp.columns else np.zeros(len(dfp))
psp = dfp['psu'].to_numpy() if 'psu' in dfp.columns else np.zeros(len(dfp))
yp = dfp['htn']

# 2. 성별 변수명 자동 감지 (male, sex, gender 중 데이터에 있는 것 자동 매칭)
gender_col = None
for c in ['male', 'sex', 'gender']:
    if c in dfp.columns:
        gender_col = c
        break

# 보정집합 구성 (age와 감지된 성별 변수)
base_adj = ['edu', 'age']
if gender_col:
    base_adj.append(gender_col)
    print(f"  -> 성별 변수 '{gender_col}' 감지 완료 (보정집합에 포함)")
else:
    print("  -> 성별 변수를 찾지 못해 age만 보정합니다.")

# 3. 무보정 및 보정 모형 실행
PEST = {}
for key, cols in [('무보정', ['edu']), ('보정', base_adj)]:
    valid_cols = [c for c in cols if c in dfp.columns]
    X = sm.add_constant(dfp[valid_cols].astype(float)).to_numpy()
    b, se, m = svyglm(yp, X, wp, stp, psp)

    o, lo, hi = or_ci(b[1], se[1])
    PEST[key] = {'cols': valid_cols, 'b': b, 'se': se, 'model': m, 'or': (o, lo, hi)}
    print(f"\n[{key}] 교육수준 OR = {o:.3f} ({lo:.3f}–{hi:.3f})   설계 기반 SE = {se[1]:.4f}  (모형 기본 SE = {np.asarray(m.bse)[1]:.4f})")

# 4. 민감도 분석 (실제 존재하는 변수만 골라서 실행)
print("\n--- 민감도 (참고용) ---")
if 'drink' in dfp.columns:
    dfp['drink_hi'] = (dfp['drink'] == 3).astype(float)

sensitivity_cases = [
    ('보정집합 + 고위험음주', base_adj + ['drink_hi'] if 'drink_hi' in dfp.columns else None),
    ('보정집합 + bmi (매개변수)', base_adj + ['bmi'] if 'bmi' in dfp.columns else None),
    ('보정집합 + prog (충돌부)', base_adj + ['prog'] if 'prog' in dfp.columns else None)
]

for lab, cols in sensitivity_cases:
    if cols is not None:
        valid_cols = [c for c in cols if c in dfp.columns]
        X = sm.add_constant(dfp[valid_cols].astype(float)).to_numpy()
        b, se, _ = svyglm(yp, X, wp, stp, psp)
        o, lo, hi = or_ci(b[1], se[1])
        print(f"{lab:36s}: OR = {o:.3f} ({lo:.3f}–{hi:.3f})")
    else:
        print(f"{lab:36s}: (변수 누락으로 스킵)")

→ 분석 대상 n = 4,962
  -> 성별 변수 'sex' 감지 완료 (보정집합에 포함)

[무보정] 교육수준 OR = 0.523 (0.489–0.560)   설계 기반 SE = 0.0345  (모형 기본 SE = 0.0345)

[보정] 교육수준 OR = 0.916 (0.843–0.995)   설계 기반 SE = 0.0422  (모형 기본 SE = 0.0422)

--- 민감도 (참고용) ---
보정집합 + 고위험음주                        : OR = 0.917 (0.844–0.996)
보정집합 + bmi (매개변수)                   : OR = 0.918 (0.844–0.998)
보정집합 + prog (충돌부)                   : (변수 누락으로 스킵)


/tmp/ipykernel_968/3490564772.py:50: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  o, lo, hi = or_ci(b[1], se[1])
/tmp/ipykernel_968/3490564772.py:52: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(f"\n[{key}] 교육수준 OR = {o:.3f} ({lo:.3f}–{hi:.3f})   설계 기반 SE = {se[1]:.4f}  (모형 기본 SE = {np.asarray(m.bse)[1]:.4f})")
/tmp/ipykernel_968/3490564772.py:50: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  o, lo, hi = or_ci(b[1], se[1])
/

In [30]:
#Table 2
# # 열: 무보정 | 보정 · 관심 노출을 맨 위에 행에

def fmt(o, lo, hi):
    return f'{o:.2f} ({lo:.2f}-{hi:.2f})'

# 관심 노출인 'edu'를 가장 맨 위에 배치하고, 나머지 보정 변수들을 아래에 배치합니다.
labels2 = {
    'edu': '교육수준 (1단계 증가)',
    'age': '연령 (1세 증가)',
    'male': '남성 (vs 여성)',
    'income': '소득 사분위 (1단계 증가)'
}

rows = []
# 보정 모형의 변수 순서대로 행을 생성하되, 'edu'를 맨 위로 오게 정렬합니다.
cols_ordered = ['edu'] + [c for c in PEST['보정']['cols'] if c != 'edu']

for j, c in enumerate(cols_ordered, start=1):
    # 인덱스 위치 찾기 (PEST['보정']['cols']에서의 위치)
    try:
        idx_pos = PEST['보정']['cols'].index(c) + 1 # 상수항 고려하여 +1
    except ValueError:
        continue

    if c == 'edu':
        # 무보정에서의 edu OR 가져오기
        un = fmt(*PEST['무보정']['or'])
    else:
        # 보정변수의 '무보정'은 단독 모형(같은 대상자)으로 각각 적합
        X1 = sm.add_constant(dfp[c].astype(float)).to_numpy()
        b1, se1, _ = svyglm(yp, X1, wp, stp, psp)
        un = fmt(*or_ci(b1[1], se1[1]))

    # 보정 모형의 OR 값 가져오기
    adj_or, adj_lo, adj_hi = or_ci(PEST['보정']['b'][idx_pos], PEST['보정']['se'][idx_pos])
    adj = fmt(adj_or, adj_lo, adj_hi)

    # 라벨 적용 (데이터에 없는 변수는 원래 이름 사용)
    row_label = labels2.get(c, c)
    rows.append({'변수': row_label, '무보정 OR (95% CI)': un, '보정 OR (95% CI)': adj})

table2_p = pd.DataFrame(rows)
print(f'Table 2. 교육수준과 고혈압의 연관성 (복합표본 로지스틱 회귀, n = {len(dfp):,})')
print(table2_p.to_string(index=False))

Table 2. 교육수준과 고혈압의 연관성 (복합표본 로지스틱 회귀, n = 4,962)
           변수  무보정 OR (95% CI)   보정 OR (95% CI)
교육수준 (1단계 증가) 0.52 (0.49-0.56) 0.92 (0.84-0.99)
   연령 (1세 증가) 1.07 (1.07-1.08) 1.07 (1.07-1.08)
          sex 1.01 (0.89-1.14) 1.07 (0.93-1.23)


/tmp/ipykernel_968/4279355082.py:36: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  adj_or, adj_lo, adj_hi = or_ci(PEST['보정']['b'][idx_pos], PEST['보정']['se'][idx_pos])
/tmp/ipykernel_968/4279355082.py:33: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  un = fmt(*or_ci(b1[1], se1[1]))
/tmp/ipykernel_968/4279355082.py:36: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  adj_or, adj_lo, adj_hi = or_ci(PEST['보정']['b'][idx_pos], PEST['보정']['se

💬Table 2 각주 — 모형 종류 · 보정변수 목록 · 설계 반영 방법(층화·집락·가중치, Taylor 선형화) · Table 2 fallacy 경고 문장

>모형 종류: 복합표본 로지스틱 회귀분석을 수행하였으며, 무보정 모형과 보정 모형(최소 보정집합: 연령, 성별)을 동일한 분석 대상자($n = 4,962$)를 기준으로 적합함

>보정변수 목록: 보정 모형은 연령(age) 및 성별(male/sex)을 보정하였으며, 민감도 분석에서는 고위험음주, 체질량지수(bmi), 충돌부 후손 변수 등을 추가로 통제함.

>설계 반영 방법: 국민건강영양조사(또는 해당 원본 자료)의 복합표본 설계를 반영하기 위해 층화변수(kstrata), 집락변수(조사구, psu), 그리고 가중치(wt)를 적용하였으며, 분산 추정에는 테일러 급수 선형화 방법을 사용함.

>Table 2 fallacy 경고: 본 표에 제시된 보정 모형의 회귀계수 및 오즈비(OR)는 관심 노출인 교육수준의 총효과(Total Effect)를 추정하기 위한 것이며, 보정변수(연령, 성별)의 계수는 인과적 효과를 의미하는 것이 아니라 교란 보정을 위한 통계적 조정값에 불과하므로 이들에 대해 개별적인 인과적 해석을 부여하는 것은 Table 2 fallacy에 해당하므로 주의해야 함.